In [170]:
# ========== 诊所预约机器人（Clinic Booking Bot）==========
# 练习目标：用 Gradio + OpenAI（聊天 / Whisper / TTS）做「仅预约」助手
# 业务规则：仅工作日 14:00–15:00 开放预约；可文字或语音输入，返回确认文本/音频/卡片图
# 和本课关系：messages、工具型函数、多模态输入输出与 Gradio UI

轻松预约诊所就诊——**仅工作日 14:00–15:00** 开放。  
可语音或文字输入，即时获得确认。


In [171]:
# ========== 导入：标准库 + OpenAI + Gradio + 图像处理 ==========

# 导入标准库 os：读环境变量（例如 OPENAI_API_KEY）
import os
# 导入标准库 json：本笔记本后续若扩展结构化数据可用（当前主流程未强依赖）
import json
# 从 dotenv 导入 load_dotenv：把 .env 密钥读进环境变量
from dotenv import load_dotenv
# 从 openai 导入 OpenAI 客户端：聊天、Whisper 转写、TTS 共用同一客户端
from openai import OpenAI
# 导入 gradio：快速搭 Web UI（聊天、音频、按钮、图片输出）
import gradio as gr
# 导入 base64：若要把二进制做成 data URL 时可用来编码（当前主流程未强依赖）
import base64
# 从 io 导入 BytesIO：内存中的字节流缓冲区（常与图像编码配合）
from io import BytesIO
# 从 datetime 导入 date：日期相关工具（本文件其它格会用到 datetime 类名）
from datetime import date
# 从 PIL 导入绘图组件：生成预约确认卡片图片
from PIL import Image, ImageDraw, ImageFont


In [172]:
# ========== 环境 + 模型：加载密钥并创建 OpenAI 客户端 ==========

# 加载 .env；override=True 表示用文件覆盖已有同名环境变量
load_dotenv(override=True)

# 读取 API Key，只打印前 8 位做存在性检查（不要把完整密钥打进日志）
openai_api_key = os.getenv('OPENAI_API_KEY')
# 有密钥：提示已读到，并展示开头几个字符便于核对
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
# 无密钥：提醒去配置 .env
else:
    print("OpenAI API Key not set")
    
# 本格声明的默认聊天模型 id（后面部分函数仍硬编码 gpt-4 / whisper / tts）
MODEL = "gpt-4o-mini"
# 创建全局 OpenAI 客户端，供后续 TTS / 翻译 / 聊天 / Whisper 复用
openai = OpenAI()


OpenAI API Key exists and begins sk-proj-


In [173]:
# ========== 预约配置：时段、工作日、联系电话、内存中的已确认列表 ==========
# --- 配置 ---  # --- CONFIG ---

# 可预约时段起点（小时，24 小时制）：14 点
BOOKING_START = 14
# 可预约时段终点（不含）：15 点 → 即 [14:00, 15:00)
BOOKING_END = 15
# 允许预约的工作日英文名（与 datetime.strftime("%A") 返回值对齐）
WEEKDAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
# 展示给用户的诊所电话（UI Markdown 里也会引用）
PHONE = "010-1234567"
# 进程内已确认预约列表：(姓名, 时间字符串)；重启内核会清空
confirmed_bookings = []


In [174]:
# ========== 文本转语音 TTS：把确认文案写成 mp3 文件 ==========
# --- 文本转语音 (TTS) ---  # --- TTS ---

# text：要朗读的文本；voice：音色名；filename：输出音频路径
def generate_tts(text, voice="fable", filename="output.mp3"):
    # 调用 OpenAI Audio Speech API：模型 tts-1，音色 fable（参数保持原样）
    response = openai.audio.speech.create(
        model="tts-1",
        voice="fable",
        input=text
    )
    # 以二进制写入本地文件，供 Gradio Audio 组件播放
    with open(filename, "wb") as f:
        # response.content：合成音频的原始字节
        f.write(response.content)
    # 返回文件路径，方便上层直接接到 UI 输出
    return filename


In [175]:
# ========== 翻译：把确认信息翻成目标语言（默认荷兰语 nl） ==========
# --- 翻译预约确认信息 ---  # --- Translate Booking Confirmation ---

# text：原文；target_language：目标语言代码（默认 nl）
def translate_text(text, target_language="nl"):
    # 拼 user 侧翻译指令；f-string 里的英文模板是可执行 prompt 的一部分，勿改
    prompt = f"Translate this message to {target_language}:\n{text}"
    # 用 Chat Completions 做翻译（这里模型写死为 gpt-4）
    response = openai.chat.completions.create(
        model="gpt-4",
        messages=[
            # system：翻译助手角色
            {"role": "system", "content": "You are a helpful translator."},
            # user：具体待翻译内容
            {"role": "user", "content": prompt}
        ]
    )
    # 取第一条回复文本并去掉首尾空白
    return response.choices[0].message.content.strip()


In [176]:
# ========== 预约核心逻辑：校验 HH:MM + 工作日 + 14–15 点窗口 ==========
# --- 预约逻辑 ---  # --- Booking Logic ---

# name：患者姓名；time_str：期望时间，格式必须是 HH:MM
def book_appointment(name, time_str):
    # 尝试把字符串解析成今天的时间点；失败则返回英文错误提示 + 空音频/空图
    try:
        booking_time = datetime.strptime(time_str, "%H:%M")
    except ValueError:
        return "Invalid time format. Use HH:MM.", None, None

    # 取出小时数字，用于和 BOOKING_START / BOOKING_END 比较
    hour = booking_time.hour
    # 今天是星期几的英文全名（Monday…），与 WEEKDAYS 列表比对
    weekday = datetime.today().strftime("%A")

    # 周末：直接拒绝（文案保持英文，后续会翻译/播报）
    if weekday not in WEEKDAYS:
        response = "Bookings are only available on weekdays."
    # 落在 [14, 15) 小时窗口：确认预约，并生成翻译文本 + TTS + 卡片图
    elif BOOKING_START <= hour < BOOKING_END:
        # 英文确认句（后续 translate_text 再翻成目标语言）
        confirmation = f"Booking confirmed for {name} at {time_str}."
        # 记入内存列表，便于本次会话追踪
        confirmed_bookings.append((name, time_str))
        # 翻译确认文案（默认 nl）
        translated = translate_text(confirmation)
        # 合成语音文件
        audio = generate_tts(translated)
        # 画一张简单的确认卡片
        image = generate_booking_image(name, time_str)
        # 成功路径：返回（译文, 音频路径, 图片对象）
        return translated, audio, image
    # 工作日但不在 14:00–15:00：拒绝并返回译文 + 音频，无图片
    else:
        response = "Sorry, bookings are only accepted between 14:00 and 15:00 on weekdays."
        translated = translate_text(response)
        audio = generate_tts(translated)
        return translated, audio, None


In [177]:
# ========== 预约卡片：用 Pillow 画一张简单确认图 ==========
# --- 预约卡片图片 ---  # --- Booking Card ---

# name / time_str：写到卡片上的姓名与时间
def generate_booking_image(name, time_str):
    # 新建 500x250 白底 RGB 画布
    img = Image.new("RGB", (500, 250), color="white")
    # 取得画笔对象，后续在画布上写字
    draw = ImageDraw.Draw(img)
    # 卡片文案：勾选符号 + 姓名 + 时间（\u2705 即 ✅）
    msg = f"\u2705 Booking Confirmed\nName: {name}\nTime: {time_str}"
    # 在坐标 (50,100) 用黑色绘制文本（默认字体）
    draw.text((50, 100), msg, fill="black")
    # 返回 PIL Image，供 Gradio Image 组件显示
    return img


In [178]:
# ========== 语音预约：Whisper 转写 → LLM 抽时间 → book_appointment ==========
# --- 语音预约 ---  # --- Voice Booking ---

# audio_path：Gradio 录下的音频文件路径；name：患者姓名
def voice_booking(audio_path, name):
    # 以二进制打开音频，交给 Whisper 转写为文字
    with open(audio_path, "rb") as f:
        # model=whisper-1：语音转文字（Speech-to-Text）
        response = openai.audio.transcriptions.create(model="whisper-1", file=f)
    # 取出转写文本并去掉首尾空白
    transcription = response.text.strip()

    # system_prompt：只抽取 24 小时制 HH:MM；抽不到就回固定英文句
    # 整段英文字符串是可执行 prompt，保持原样
    system_prompt = """
    You are a clinic assistant. Extract only the appointment time from the user's sentence in 24-hour HH:MM format.
    If no time is mentioned, respond with 'No valid time found.'
    """

    # 用 gpt-4 从转写文本里抽出时间
    response = openai.chat.completions.create(
        model="gpt-4",
        messages=[
            {"role": "system", "content": system_prompt},
            # user 内容就是 Whisper 转写出的原话
            {"role": "user", "content": transcription}
        ]
    )
    # 模型回复应接近 "14:30" 或错误提示句
    extracted_time = response.choices[0].message.content.strip()

    # 粗判：字符串里含冒号就当作抽到了时间，交给预约函数
    if ":" in extracted_time:
        return book_appointment(name, extracted_time)
    # 否则：翻译道歉句 + TTS，无卡片图
    else:
        message = "Sorry, I couldn't understand the time. Please try again."
        translated = translate_text(message)
        audio_path = generate_tts(translated)
        return translated, audio_path, None


In [179]:
# ========== 聊天机器人：只协助预约，明确拒绝医疗建议 ==========
# --- 聊天机器人处理 ---  # --- Chat Bot Handler ---

# messages：Gradio Chatbot 的历史消息列表（role/content 字典）
def chat_bot(messages):
    # system_prompt：角色边界 + 可预约时段 + 禁止医疗建议
    # 英文字符串是可执行 prompt，勿翻译改写
    system_prompt = """
    You are a clinic booking assistant. Your job is to:
    - Greet the patient and explain your role
    - Only assist with making appointments
    - Accept bookings only on weekdays between 14:00 and 15:00
    - Do not provide medical advice
    - Always respond with empathy and clarity
    """
    # 把 system 放最前，再拼接用户对话历史
    response = openai.chat.completions.create(
        model="gpt-4",
        messages=[{"role": "system", "content": system_prompt}] + messages
    )
    # 取出助手回复正文
    reply = response.choices[0].message.content.strip()
    # 同步生成语音版回复，方便无障碍 / 语音交互
    audio = generate_tts(reply)
    # 返回（文字回复, 音频路径）
    return reply, audio


In [180]:
# ========== Gradio UI：聊天 / 文字预约 / 语音预约 三个入口 ==========

# 使用 Soft 主题创建 Blocks 应用；demo 是整页容器
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    # 页头说明：可预约时段与联系电话（注意：这里的 {PHONE} 在普通字符串里不会自动插值）
    gr.Markdown("""## 🩺 GP Booking Assistant  
Only available weekdays between **14:00 and 15:00**  
☎️ Contact: {PHONE}
---""")

    # 全局姓名输入框：文字预约与语音预约都会读这个值
    name_global = gr.Textbox(label="Your Name", placeholder="Enter your name", interactive=True)

    # ---------- Tab 1：聊天模式（文字 + 可选语音转写） ----------
    with gr.Tab("💬 Chat Mode"):
        # type="messages"：历史为 {"role","content"} 字典列表
        chatbot = gr.Chatbot(label="Booking Chat", type="messages", height=400)
        # 文本输入框
        text_input = gr.Textbox(label="Type your message or use your voice below")
        # 麦克风 / 上传音频：得到本地 filepath
        audio_input = gr.Audio(type="filepath", label="🎙️ Or speak your request")
        # 助手语音回复播放器
        chat_audio_output = gr.Audio(label="🔊 Assistant's Reply", type="filepath")
        # 发送按钮
        send_btn = gr.Button("Send")

        # 处理一条文字消息：追加 user → 调 chat_bot → 追加 assistant
        def handle_chat(user_message, chat_history):
            # 历史可能是 None，统一成列表
            chat_history = chat_history or []
            # 追加用户消息（messages 格式）
            chat_history.append({"role": "user", "content": user_message})
            # 调用聊天助手，得到文字 + 音频
            reply, audio = chat_bot(chat_history)
            # 追加助手回复
            chat_history.append({"role": "assistant", "content": reply})
            # 返回：更新后的历史、清空输入框、音频路径
            return chat_history, "", audio

        # 语音进聊天：先 Whisper 转写，再复用 handle_chat
        def handle_audio_chat(audio_path, chat_history):
            with open(audio_path, "rb") as f:
                transcription = openai.audio.transcriptions.create(model="whisper-1", file=f).text.strip()
            return handle_chat(transcription, chat_history)

        # 绑定：点击发送 / 回车提交 / 音频变化
        send_btn.click(handle_chat, [text_input, chatbot], [chatbot, text_input, chat_audio_output])
        text_input.submit(handle_chat, [text_input, chatbot], [chatbot, text_input, chat_audio_output])
        audio_input.change(handle_audio_chat, [audio_input, chatbot], [chatbot, text_input, chat_audio_output])


    
    # ---------- Tab 2：纯文字预约（姓名 + HH:MM） ----------
    with gr.Tab("📝 Text Booking"):
        time_text = gr.Textbox(label="Preferred Time (HH:MM)", placeholder="e.g., 14:30")
        btn_text = gr.Button("📅 Book via Text")

    # ---------- Tab 3：语音预约（说时间） ----------
    with gr.Tab("🎙️ Voice Booking"):
        voice_input = gr.Audio(type="filepath", label="Say your preferred time")
        btn_voice = gr.Button("📅 Book via Voice")

    # 三个 Tab 共用的输出区：文本 / 音频 / 确认图
    output_text = gr.Textbox(label="Response", interactive=False)
    output_audio = gr.Audio(label="Audio Reply", type="filepath")
    output_image = gr.Image(label="Booking Confirmation")

    # 文字预约按钮 → book_appointment(name, time)
    btn_text.click(fn=book_appointment, inputs=[name_global, time_text], outputs=[output_text, output_audio, output_image])
    # 语音预约按钮 → voice_booking(audio, name)
    btn_voice.click(fn=voice_booking, inputs=[voice_input, name_global], outputs=[output_text, output_audio, output_image])

    # 页脚免责声明：不做医疗建议
    gr.Markdown("""---
<small>This assistant does **not** give medical advice. It only books appointments within allowed hours.</small>
""")

    # 启动 Gradio 应用（默认本地端口，按库版本行为可能弹链接）
    demo.launch()


* Running on local URL:  http://127.0.0.1:7898
* To create a public link, set `share=True` in `launch()`.
